# CNNs for Medical Imaging: Finding Pneumonia in Chest X-rays (Easy Version)

**Goes with:** the slide deck *CNNs for Medical Imaging: Teaching a Computer to Read Chest X-rays*.

This is the **beginner-friendly** version of the project. It does the same job as the *Hard* notebook, but it lets ready-made library tools do the heavy lifting, so each code cell stays short.

| Notebook part | What we do | Slides |
|---|---|---|
| 1. Data | Download and explore a real, open dataset of children's chest X-rays | Part 3 |
| 2. Prepare | Set up data augmentation | Part 4 |
| 3. Build | Design a convolutional neural network (CNN) | Part 4 |
| 4. Train | Train it with class weights and early stopping | Part 4 |
| 5. Evaluate | Confusion matrix, sensitivity, specificity, thresholds, ROC and AUC | Part 5 |
| 6. Explain | Learned filters, feature maps, Grad-CAM heat maps, and mistakes | Part 6 |

> ⚠️ **Important:** This is a **learning project**, not a medical tool. It must never be used to make decisions about anyone's health. Real medical AI needs years of testing, doctors in charge, and approval from health regulators (slides, Part 7).

**How to run:** This notebook works best in **Google Colab** (free). Choose *Runtime → Change runtime type → GPU* to make training faster (optional). Then run each cell from top to bottom with **Shift + Enter**. Training takes about 2 to 5 minutes.

**Tip:** Cells marked **🔧 Helper cell** contain drawing code. Just run them; you do not need to understand every line.

## 0. Setup

We load the libraries (ready-made tools) we need. All of them are free and open source.

In [ ]:
import numpy as np                    # numbers and grids of numbers
import pandas as pd                   # neat tables
import matplotlib.pyplot as plt       # charts and pictures
import tensorflow as tf               # the deep learning engine
from tensorflow import keras          # the easy-to-use part of TensorFlow
from tensorflow.keras import layers   # building blocks: Conv2D, Dense, ...

# Ready-made tools from scikit-learn for measuring how good our model is.
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
                             roc_curve, roc_auc_score, recall_score, accuracy_score)

keras.utils.set_random_seed(42)       # get (almost) the same result every time we run
print("TensorFlow version:", tf.__version__)

---
# Part 1 · The Data (slides, Part 3)

## 1.1 Download the dataset

We use **PneumoniaMNIST**, part of the open **MedMNIST v2** collection.

* **What it is:** 5,856 chest X-rays of young children, labeled **normal** (0) or **pneumonia** (1).
* **Image size:** 64 × 64 pixels, grayscale.
* **License:** CC BY 4.0 (free to use, as long as we give credit).

The file already contains three separate piles (slide 16):

* **Training set:** practice problems. The network learns from these.
* **Validation set:** practice quizzes. We use them to choose the best epoch and the threshold.
* **Test set:** the final exam. We use it **only once**, at the very end.

In [ ]:
# Download the file once (about 20 MB). Running this cell again uses the saved copy.
url = "https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1"
path = keras.utils.get_file("pneumoniamnist_64.npz", origin=url)
data = np.load(path)

# Unpack the three piles. ravel() turns each list of labels into one flat row.
x_train, y_train = data["train_images"], data["train_labels"].ravel()
x_val,   y_val   = data["val_images"],   data["val_labels"].ravel()
x_test,  y_test  = data["test_images"],  data["test_labels"].ravel()

CLASS_NAMES = ["NORMAL", "PNEUMONIA"]   # label 0, label 1

print("Training:  ", len(y_train), "X-rays")
print("Validation:", len(y_val), "X-rays")
print("Test:      ", len(y_test), "X-rays")

## 1.2 Look at the X-rays (slide 14)

Always look at your data before training! Can **you** spot the difference between the two rows?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for label in [0, 1]:                          # row 0 = normal, row 1 = pneumonia
    examples = x_train[y_train == label]      # keep only the X-rays with this label
    for col in range(5):
        axes[label, col].imshow(examples[col], cmap="gray")
        axes[label, col].set_title(CLASS_NAMES[label])
        axes[label, col].axis("off")
plt.tight_layout()
plt.show()

## 1.3 An X-ray is just numbers (slide 15)

Each pixel is one number from **0 (black)** to **255 (white)**. Let us print a small 8 × 8 patch.

In [ ]:
image = x_train[0]
print("Size of one X-ray:", image.shape, "=", image.size, "numbers")
print(image[28:36, 10:18])      # rows 28 to 35, columns 10 to 17

## 1.4 The hidden trap: unbalanced data (slide 17)

About 3 out of 4 training X-rays show pneumonia. A "lazy" model that **always** answers "pneumonia" would still get a decent accuracy, without looking at a single X-ray.

In [ ]:
print(f"Training X-rays with pneumonia: {np.mean(y_train == 1):.0%}")

# The lazy model says "pneumonia" (1) every time, so it is right exactly when the answer is 1.
lazy_accuracy = np.mean(y_test == 1)
print(f"Lazy model accuracy on the test set: {lazy_accuracy:.0%}")
print("Our CNN must do MUCH better than this to be useful.")

---
# Part 2 · Prepare the Images (slides 19 and 20)

## 2.1 Data augmentation: more variety for free (slide 20)

Each time the network sees a training X-ray, we change it a little:

* **Rotate** by up to about 11° (`0.03` of a full turn)
* **Shift** up and down or left and right by up to 5%
* **Zoom** in or out by up to 10%

We **never flip** chest X-rays left to right: the heart is on the left side, so a flipped chest would not look real.

These layers are only active **during training**. During testing, they do nothing.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomRotation(0.03),             # turn a little (up to about 11°)
    layers.RandomTranslation(0.05, 0.05),    # shift up to 5%
    layers.RandomZoom(0.1),                  # zoom in or out up to 10%
])

# Show one X-ray and 5 random changed versions of it.
original = x_train[0].reshape(1, 64, 64, 1)  # Keras wants: (1 image, 64 rows, 64 columns, 1 channel)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(x_train[0], cmap="gray")
axes[0].set_title("Original")
for k in range(1, 6):
    changed = augmentation(original, training=True)       # training=True switches it ON
    axes[k].imshow(np.array(changed).reshape(64, 64), cmap="gray")
    axes[k].set_title(f"Version {k}")
for ax in axes:
    ax.axis("off")
plt.show()

---
# Part 3 · Build the CNN (slide 21)

We stack layers like LEGO bricks with `keras.Sequential`. The first two layers prepare each X-ray:

* **Reshape** adds a **channel**. Color photos have 3 channels (red, green, blue); X-rays have 1. So each image becomes 64 × 64 × **1**.
* **Rescaling** divides every pixel by 255, so values go from **0 to 1**. Networks learn best with small numbers.

Then comes the classic pattern: **Conv → Pool → Conv → Pool → Conv → Pool**, then **Flatten → Dropout → Dense → Output**.

| Layer | What it does | Output shape |
|---|---|---|
| Conv2D (16 filters) + ReLU | Finds 16 simple patterns, like edges | 64 × 64 × 16 |
| MaxPooling | Halves the size, keeps the strongest signals | 32 × 32 × 16 |
| Conv2D (32 filters) + ReLU | Combines edges into shapes | 32 × 32 × 32 |
| MaxPooling | Halves the size again | 16 × 16 × 32 |
| Conv2D (64 filters) + ReLU | Finds larger patterns, like hazy areas | 16 × 16 × 64 |
| MaxPooling | Halves the size again | 8 × 8 × 64 |
| Flatten | Unrolls the grid into one list | 4,096 |
| Dropout (30%) | Randomly switches off 30% of the values during training | 4,096 |
| Dense (32) + ReLU | Combines all the features | 32 |
| Dense (1) + sigmoid | The **chance of disease**, from 0 to 1 | 1 |

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(64, 64)),           # one grayscale X-ray
    layers.Reshape((64, 64, 1)),           # add the 1 channel
    layers.Rescaling(1 / 255),             # pixels 0..255  ->  0..1
    augmentation,                          # random changes (training only)

    layers.Conv2D(16, 3, padding="same", activation="relu", name="conv1"),
    layers.MaxPooling2D(),                 # 64x64 -> 32x32
    layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2"),
    layers.MaxPooling2D(),                 # 32x32 -> 16x16
    layers.Conv2D(64, 3, padding="same", activation="relu", name="conv3"),
    layers.MaxPooling2D(),                 # 16x16 -> 8x8

    layers.Flatten(),                      # 8 x 8 x 64 = 4,096 numbers
    layers.Dropout(0.3),                   # fight memorizing (slide 22)
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"), # chance of disease, 0 to 1
])
model.summary()

**Where do the parameter counts come from?** Take `conv1`: each of its 16 filters has 3 × 3 = 9 weights, plus 1 bias. That makes 16 × (9 + 1) = **160** knobs. Training will tune all of them automatically.

---
# Part 4 · Train the CNN (slides 22 and 23)

## 4.1 Class weights (slide 17)

To fix the unbalanced data, mistakes on the smaller class must count more. The usual formula is:

$$\text{weight of a class} = \frac{\text{total number of images}}{2 \times \text{number of images in that class}}$$

scikit-learn calculates this for us when we ask for `"balanced"` weights.

In [ ]:
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}

print(f"Weight for NORMAL:    {class_weight[0]:.2f}  (the smaller class counts more)")
print(f"Weight for PNEUMONIA: {class_weight[1]:.2f}")

## 4.2 Compile: choose how the network learns

* **Optimizer `adam`:** a smart version of gradient descent that adjusts its own step size.
* **Loss `binary_crossentropy`:** the standard loss for yes/no questions. Lower is better.
* **Metrics:** accuracy and **AUC** (slide 31).

In [ ]:
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])

## 4.3 Train with early stopping (slides 22 and 23)

* **`epochs=25`:** at most 25 passes through the training X-rays.
* **`batch_size=64`:** update the weights after every group of 64 X-rays.
* **Early stopping:** after each epoch, we check the **validation AUC**. If it does not improve for 6 epochs in a row, training stops, and we go back to the **best** epoch.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max",   # watch the validation AUC; higher is better
    patience=6,                      # stop after 6 epochs without improvement
    restore_best_weights=True)       # then go back to the best epoch

history = model.fit(x_train, y_train,
                    validation_data=(x_val, y_val),   # the "practice quiz" after every epoch
                    epochs=25,
                    batch_size=64,
                    class_weight=class_weight,        # our fix for unbalanced data
                    callbacks=[early_stopping],
                    verbose=2)                        # print one line per epoch

## 4.4 Watch it train (slide 23)

In [ ]:
results = pd.DataFrame(history.history)      # one row per epoch
results.index += 1                           # count epochs from 1, not 0

results[["loss", "val_loss"]].plot(title="Loss goes down", xlabel="epoch", marker="o", grid=True)
results[["auc", "val_auc"]].plot(title="AUC goes up", xlabel="epoch", marker="o", grid=True)
plt.show()

**Two things to notice (slide 23):**

* The validation curve is **bumpy**. The validation set is small, so a few changed answers move the score a lot.
* Validation can look **better** than training. That is because dropout and augmentation make training harder on purpose, and they are switched off for validation.

---
# Part 5 · Evaluate Like a Doctor (slides 25 to 32)

## 5.1 Ask the network about the test X-rays

For each X-ray, the network outputs a **chance** from 0 to 1. We decide with a **threshold**: at or above it, we say "disease"; below it, "normal". We start with the usual threshold, **0.5**.

In [ ]:
test_chance = model.predict(x_test, verbose=0).ravel()   # one chance per test X-ray
val_chance = model.predict(x_val, verbose=0).ravel()     # one chance per validation X-ray

test_answer = (test_chance >= 0.5).astype(int)           # True/False -> 1/0
print(f"Test accuracy: {accuracy_score(y_test, test_answer):.1%}   (lazy model: {lazy_accuracy:.1%})")

## 5.2 The confusion matrix (slide 26)

Rows show the **truth**. Columns show what the **network said**.

| | said NORMAL | said PNEUMONIA |
|---|---|---|
| **really NORMAL** | correct (TN) | false alarm (FP) |
| **really PNEUMONIA** | **MISSED** (FN) | caught (TP) |

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, test_answer, display_labels=CLASS_NAMES, cmap="Blues")
plt.title("Test set, threshold 0.5")
plt.show()

## 5.3 Sensitivity and specificity (slide 27)

* **Sensitivity** = caught ÷ all sick = TP ÷ (TP + FN). *How many sick people did we find?*
* **Specificity** = cleared ÷ all healthy = TN ÷ (TN + FP). *How many healthy people did we clear?*

scikit-learn calls both of these **recall**: sensitivity is the recall of the PNEUMONIA class, and specificity is the recall of the NORMAL class. The report below shows them in the `recall` column.

For a **screening tool**, sensitivity matters most: missing a sick child is worse than a false alarm.

In [ ]:
print(classification_report(y_test, test_answer, target_names=CLASS_NAMES))

## 5.4 The network gives a chance, not an answer (slide 28)

This chart shows the chance the network gave to every test X-ray. Most are easy: very close to 0 or to 1. The hard ones sit in the middle, close to the threshold.

In [ ]:
plt.hist(test_chance[y_test == 0], bins=25, alpha=0.7, label="really NORMAL")
plt.hist(test_chance[y_test == 1], bins=25, alpha=0.7, label="really PNEUMONIA")
plt.axvline(0.5, color="black", label="threshold 0.5")
plt.xlabel("the network's chance of disease")
plt.ylabel("number of test X-rays")
plt.legend()
plt.show()

## 5.5 Moving the threshold (slide 29)

What happens if we move the threshold? Lower thresholds catch more sick children but cause more false alarms.

This small function scores the test answers for **any** threshold we give it.

In [ ]:
def check(threshold):
    answer = (test_chance >= threshold).astype(int)
    return {"threshold": threshold,
            "sensitivity": recall_score(y_test, answer, pos_label=1),   # sick cases caught
            "specificity": recall_score(y_test, answer, pos_label=0),   # healthy cases cleared
            "accuracy": accuracy_score(y_test, answer)}

pd.DataFrame([check(t) for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]]).round(3)

## 5.6 Choosing a threshold, the careful way (slide 30)

**Our rule:** catch at least **95%** of sick cases.

We must choose the threshold using the **validation set**, not the test set. If we chose it by looking at the test set, our final score would no longer be honest.

`roc_curve` tries every possible threshold for us, from high to low, and tells us the sensitivity (the "catch rate") at each one. We pick the **first** (highest) threshold that reaches 95%: it reaches the goal with as few false alarms as possible.

In [ ]:
false_alarm_rate, catch_rate, thresholds = roc_curve(y_val, val_chance)
first_good = np.argmax(catch_rate >= 0.95)      # position of the FIRST threshold that catches 95%
chosen_threshold = thresholds[first_good]
print(f"Threshold chosen on the VALIDATION set: {chosen_threshold:.2f}")

# Only now do we check the result on the test set.
pd.DataFrame([check(0.5), check(chosen_threshold)]).round(3)

In [ ]:
# Sensitivity and specificity for 99 thresholds between 0.01 and 0.99 (on the test set).
curves = pd.DataFrame([check(t) for t in np.linspace(0.01, 0.99, 99)])
curves.plot(x="threshold", y=["sensitivity", "specificity"], grid=True)
plt.axvline(0.5, color="black", linestyle="--", label="default 0.5")
plt.axvline(chosen_threshold, color="red", linestyle="--", label="our choice")
plt.legend()
plt.show()

## 5.7 The ROC curve and AUC (slide 31)

The ROC curve shows **every** threshold at once. AUC is the **area under the curve**: 0.5 is a coin flip, and 1.0 is perfect.

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_chance, name="our CNN", plot_chance_level=True)
plt.show()

print(f"Validation AUC: {roc_auc_score(y_val, val_chance):.3f}")
print(f"Test AUC:       {roc_auc_score(y_test, test_chance):.3f}")

**A warning sign (slide 32):** the test AUC is usually **lower** than the validation AUC. In PneumoniaMNIST, the test X-rays come from a different part of the original collection than the training X-rays. New data is often a little different, and real hospitals are even more different. Medical AI must be tested on data from many places.

---
# Part 6 · Opening the Black Box (slides 34 to 37)

## 6.1 What did the first layer learn? (slide 34)

These 16 filters started as random numbers. Training shaped them into pattern detectors.

In [ ]:
filters = model.get_layer("conv1").get_weights()[0]    # shape (3, 3, 1, 16): 16 filters of 3 x 3

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for k, ax in enumerate(axes.flat):
    ax.imshow(filters[:, :, 0, k], cmap="coolwarm")    # red = positive, blue = negative
    ax.set_title(f"filter {k + 1}")
    ax.axis("off")
plt.show()

### Feature maps: one X-ray through the learned filters

We make a small "spy" model that stops after `conv1`, so we can see what each filter found.

In [ ]:
spy_model = keras.Model(inputs=model.inputs, outputs=model.get_layer("conv1").output)

xray = x_test[y_test == 1][4]                                 # one pneumonia X-ray
feature_maps = spy_model.predict(xray[np.newaxis], verbose=0)[0]   # shape (64, 64, 16)

fig, axes = plt.subplots(2, 9, figsize=(16, 4.5))
axes.flat[0].imshow(xray, cmap="gray")
axes.flat[0].set_title("input")
for k in range(16):
    axes.flat[k + 1].imshow(feature_maps[:, :, k], cmap="magma")
    axes.flat[k + 1].set_title(f"map {k + 1}")
for ax in axes.flat:
    ax.axis("off")
plt.show()

**Bright areas** show where each filter found its pattern. **Completely dark maps** did not find their pattern anywhere in this X-ray: ReLU turned all their negative values into 0.

## 6.2 Grad-CAM: where is the network looking? (slide 35)

**Grad-CAM** builds a heat map. In short, it asks: *"Which parts of the last feature maps (`conv3`) pushed the disease score up the most?"* Warm colors (red, yellow) show the areas that pushed the network **toward disease**.

The cell below defines `explain_xray()`, which shows one test X-ray, its heat map, and the network's answer.

In [ ]:
# 🔧 Helper cell: just run it. You do not need to understand every line.

LAST_CONV = "conv3"   # the conv layer that Grad-CAM looks inside

def grad_cam(image):
    # A model with TWO outputs: the feature maps of LAST_CONV and the final chance.
    grad_model = keras.Model(model.inputs, [model.get_layer(LAST_CONV).output, model.layers[-1].output])
    with tf.GradientTape() as tape:
        maps, chance = grad_model(image[np.newaxis], training=False)
        chance = tf.clip_by_value(chance, 1e-7, 1 - 1e-7)
        score = tf.math.log(chance / (1 - chance))      # the score before the sigmoid squashed it
    gradients = tape.gradient(score, maps)[0]           # how much each map pixel moves the score
    importance = tf.reduce_mean(gradients, axis=(0, 1)) # one weight per feature map
    heatmap = tf.nn.relu(tf.reduce_sum(maps[0] * importance, axis=-1)).numpy()
    heatmap = heatmap / (heatmap.max() + 1e-8)          # scale to 0..1
    return tf.image.resize(heatmap[..., np.newaxis], (64, 64)).numpy()[..., 0]

def explain_xray(index, threshold=0.5):
    image = x_test[index]
    chance = test_chance[index]
    answer = CLASS_NAMES[int(chance >= threshold)]
    truth = CLASS_NAMES[y_test[index]]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title(f"Test X-ray #{index}")
    axes[1].imshow(image, cmap="gray")
    axes[1].imshow(grad_cam(image), cmap="jet", alpha=0.45)   # heat map on top
    axes[1].set_title("Grad-CAM heat map")
    for ax in axes[:2]:
        ax.axis("off")
    axes[2].barh(CLASS_NAMES, [1 - chance, chance], color=["tab:blue", "tab:red"])
    axes[2].axvline(threshold, color="orange", linestyle="--", label=f"threshold {threshold:.2f}")
    axes[2].set_xlim(0, 1)
    axes[2].legend(loc="lower right")
    axes[2].set_title(f"Network: {answer}  |  Truth: {truth}",
                      color="green" if answer == truth else "red")
    plt.tight_layout()
    plt.show()

print("explain_xray() is ready.")

In [ ]:
explain_xray(2)
explain_xray(11)

**Be careful with heat maps (slide 37).** If the warm area sits on a text marker, a wire, or the edge of the picture instead of the lungs, the network may have learned a **shortcut**. A high score is not proof: we must check *why* the model is right.

## 6.3 Learning from mistakes (slide 36)

`np.where` finds the positions of all the X-rays that match a condition.

In [ ]:
missed = np.where((y_test == 1) & (test_answer == 0))[0]        # sick, but we said normal
false_alarms = np.where((y_test == 0) & (test_answer == 1))[0]  # healthy, but we said sick
print(f"Missed: {len(missed)}   False alarms: {len(false_alarms)}")

for index in missed[:2]:
    explain_xray(index)
for index in false_alarms[:2]:
    explain_xray(index)

## 6.4 Ask the network about one X-ray

This is how a doctor-facing tool *might* present a result: the chance, the answer at **our chosen threshold**, and a heat map. Remember: this is a **learning project**, not a diagnosis.

In [ ]:
# ===== CHANGE THIS NUMBER TO LOOK AT A DIFFERENT TEST X-RAY (0 to 623) =====
explain_xray(11, threshold=chosen_threshold)

---
# Summary

| Step | What we did | Key idea |
|---|---|---|
| Data | Loaded 5,856 open-access chest X-rays | Keep training, validation, and test sets apart |
| Prepare | Added augmentation | Never flip chest X-rays |
| Build | A CNN with 3 Conv blocks | Picture shrinks, feature maps grow |
| Train | Class weights, dropout, early stopping | Fight unbalanced data and memorizing |
| Evaluate | Confusion matrix, sensitivity, specificity, ROC | The threshold is a human choice |
| Explain | Filters, feature maps, Grad-CAM, mistakes | Check *why* the model is right |

## Challenge questions

1. **Change the network:** delete the third Conv block (the `conv3` line and the `MaxPooling2D()` after it). In the Grad-CAM helper cell, change `LAST_CONV = "conv3"` to `"conv2"`. Does the test AUC go up or down?
2. **Turn off augmentation:** delete the `augmentation,` line from the model. What happens to the gap between validation and test AUC?
3. **Pick your threshold:** change `0.95` to `0.99` in section 5.6. How many false alarms does it cost to catch almost every sick child?
4. **New dataset:** in section 1.1, change both `pneumoniamnist_64.npz` names to `breastmnist_64.npz`, and set `CLASS_NAMES = ["NORMAL or BENIGN", "MALIGNANT"]`. In BreastMNIST, 0 means malignant, so also add this line at the end of that cell to flip the labels: `y_train, y_val, y_test = 1 - y_train, 1 - y_val, 1 - y_test`. Run everything again. Is it easier or harder? (Hint: compare the number of training images.)
5. **Bigger challenge:** MedMNIST also has **BloodMNIST** (8 types of blood cells) and **OrganAMNIST** (11 organs from CT scans). They have many classes, so the output must become `Dense(number_of_classes, activation="softmax")`, and the loss must become `sparse_categorical_crossentropy`. Find them at medmnist.com.

## Dataset credits

* Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., & Ni, B. (2023). MedMNIST v2: A large-scale lightweight benchmark for 2D and 3D biomedical image classification. *Scientific Data*, 10, 41. License: CC BY 4.0.
* Kermany, D. S., Goldbaum, M., et al. (2018). Identifying medical diagnoses and treatable diseases by image-based deep learning. *Cell*, 172(5), 1122–1131.